In [1]:
import pandas as pd

df = pd.read_csv("/content/CTS_CRM_Objection_Annotated_3000_ContextAware.csv")

print(df.shape)
df.head()

(3000, 20)


,interaction_id,crm_note_id,interaction_date,rep_id,hcp_id,city,region,territory_id,hcp_specialization,therapeutic_area,drug_id,drug_name,brand_name,crm_note,clean_text,objection_label,annotation_confidence,annotation_rationale,annotation_method,annotation_status
0,INT003159,CRM003159,2026-08-11,REP0036,HCP00676,Chennai,South,CHENNAI_CENTRAL,Dermatologist,Dermatology,DRG008,Clearvex,Clearvanta,The representative followed up on the HCP's re...,representative follow hcp recent experience cl...,Dosing / Administration,medium,regimen,Context-aware machine-assisted annotation,Candidate label; use as training seed after sp...
1,INT011617,CRM011617,2026-02-06,REP0039,HCP02493,Chennai,South,CHENNAI_CENTRAL,General Physician,Primary Care,DRG001,Acetovarin,Relivanta,Quick f/u on Relivanta. new starts are getting...,quick follow relivanta new start get stick app...,Access / Reimbursement,medium,access; approval,Context-aware machine-assisted annotation,Candidate label; use as training seed after sp...
2,INT014453,CRM014453,2026-07-29,REP0112,HCP03083,Kolkata,East,KOLKATA_SOUTH,Nephrologist,Nephrology,DRG020,Nephrel,Nephrelis,The conversation focused on how Nephrelis is f...,conversation focus nephrelis fitting current p...,Dosing / Administration,medium,regimen,Context-aware machine-assisted annotation,Candidate label; use as training seed after sp...
3,INT012609,CRM012609,2026-06-23,REP0112,HCP02708,Kolkata,East,KOLKATA_SOUTH,Endocrinologist,Endocrinology,DRG013,Glycetrel,Glycetra,Follow-up discussion on Glycetra. copay is sto...,follow discussion glycetra copay stop start hc...,Cost / Affordability,high,copay; affordability,Context-aware machine-assisted annotation,Candidate label; use as training seed after sp...
4,INT009860,CRM009860,2026-04-20,REP0011,HCP02113,Bengaluru,South,BENGALURU_CENTRAL,General Physician,Primary Care,DRG001,Acetovarin,Relivanta,Saw HCP re Relivanta. several patients are not...,see hcp regard relivanta several patient not f...,Adherence,high,not following the regimen; not following; regi...,Context-aware machine-assisted annotation,Candidate label; use as training seed after sp...


In [2]:
#Define X and y
X = df["clean_text"]
y = df["objection_label"]

In [3]:
#train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", len(X_train))
print("Testing :", len(X_test))

Training: 2400
Testing : 600


In [4]:
#TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

In [5]:
#Fit TF-IDF only on training data
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape :", X_test_tfidf.shape)

Training TF-IDF shape: (2400, 3387)
Testing TF-IDF shape : (600, 3387)


In [6]:
#Train Linear SVM
from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    random_state=42
)

svm_model.fit(X_train_tfidf, y_train)

print("Linear SVM training completed!")

Linear SVM training completed!


In [7]:
#Make predictions
y_pred_svm = svm_model.predict(X_test_tfidf)

print(y_pred_svm[:10])

['Dosing / Administration' 'Efficacy / Clinical Benefit' 'Adherence'
 'Efficacy / Clinical Benefit' 'Access / Reimbursement'
 'Evidence / Clinical Data' 'Safety / Tolerability'
 'Evidence / Clinical Data' 'Cost / Affordability'
 'Dosing / Administration']


In [8]:
#accuracy
from sklearn.metrics import accuracy_score

svm_accuracy = accuracy_score(y_test, y_pred_svm)

print("Linear SVM Accuracy:", round(svm_accuracy, 4))

Linear SVM Accuracy: 0.93


In [9]:
#Classification report
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_svm))

                             precision    recall  f1-score   support

     Access / Reimbursement       0.96      0.97      0.96        66
                  Adherence       0.99      1.00      0.99        67
                 Competitor       0.92      0.98      0.95        66
       Cost / Affordability       0.94      0.94      0.94        67
    Dosing / Administration       0.90      0.99      0.94        67
Efficacy / Clinical Benefit       0.90      0.85      0.88        67
   Evidence / Clinical Data       0.95      0.91      0.93        66
               No Objection       0.87      0.82      0.85        67
      Safety / Tolerability       0.94      0.91      0.92        67

                   accuracy                           0.93       600
                  macro avg       0.93      0.93      0.93       600
               weighted avg       0.93      0.93      0.93       600



In [10]:
#Macro-F1
from sklearn.metrics import f1_score

svm_macro_f1 = f1_score(
    y_test,
    y_pred_svm,
    average="macro"
)

print("Linear SVM Macro-F1:", round(svm_macro_f1, 4))

Linear SVM Macro-F1: 0.9294


In [11]:
import joblib

joblib.dump(svm_model, "objection_svm_model.joblib")
joblib.dump(tfidf, "objection_tfidf_vectorizer.joblib")

print("Models saved successfully!")

Models saved successfully!
